In [2]:
!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

dataset = fetch_ucirepo(id=468)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)

print(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df['Revenue'].value_counts())

   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb                 1   
1         0.00       0.10         0.0         0.0   Feb   

In [3]:
print(df.info())
print(df.isnull().sum().sum())
print(df['Revenue'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [4]:
X = df.drop(columns=['Revenue'])
y = df['Revenue'].astype(int)

print("Features:", X.shape)
print("Target classes:", y.unique())

Features: (12330, 17)
Target classes: [0 1]


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [7]:
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

Accuracy: 88.12 %

Confusion Matrix:
 [[2037   47]
 [ 246  136]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.98      0.93      2084
           1       0.74      0.36      0.48       382

    accuracy                           0.88      2466
   macro avg       0.82      0.67      0.71      2466
weighted avg       0.87      0.88      0.86      2466

ROC-AUC: 0.8876


In [8]:
customer = X_test.iloc[[0]]

probabilities = model.predict_proba(customer)[0]

print("Purchase probability:", round(probabilities[1] * 100, 2), "%")
print("No-purchase probability:", round(probabilities[0] * 100, 2), "%")

print("Prediction:",
      "Likely to purchase" if model.predict(customer)[0] == 1
      else "Unlikely to purchase")

Purchase probability: 2.28 %
No-purchase probability: 97.72 %
Prediction: Unlikely to purchase


In [9]:
print("Logistic Regression was used for online purchase prediction.")
print("The model predicts whether a browsing session results in a purchase.")
print("Accuracy, confusion matrix, classification report and ROC-AUC")
print("were used to evaluate the model.")

Logistic Regression was used for online purchase prediction.
The model predicts whether a browsing session results in a purchase.
Accuracy, confusion matrix, classification report and ROC-AUC
were used to evaluate the model.
